In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np
import pandas as pd

In [2]:
## Load the trained model,scaler pickel ,one hot
model = load_model('model.h5')


## load the encoder and scaler
with open('label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)

with open('onehot_encoder.pkl', 'rb') as file:
    onehot_encoder = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [3]:
#Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [9]:
import pandas as pd


geography_input = pd.DataFrame([[input_data['Geography']]], columns=['Geography'])


geography_encoded = onehot_encoder.transform(geography_input)


if hasattr(geography_encoded, "toarray"):
    geography_encoded = geography_encoded.toarray()


geography_df = pd.DataFrame(
    geography_encoded, 
    columns=onehot_encoder.get_feature_names_out(['Geography'])
)

geography_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [12]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [13]:
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [14]:
## concatenate the one-hot encoded geography columns with the input_df
input_df = pd.concat([input_df.drop('Geography', axis=1), geography_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [15]:
## scale the input data using the loaded scaler
scaled_input = scaler.transform(input_df)
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [16]:
##Predict the output using the trained model
model_prediction = model.predict(scaled_input)
model_prediction

1/1 [==============================] - 0s 307ms/step


array([[0.0479739]], dtype=float32)

In [17]:
prediction_probability = model_prediction[0][0]

In [18]:
prediction_probability

0.047973905

In [20]:
if prediction_probability > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
